# Capítulo 4 — Regressão e Previsão

**Livro:** começa na página **175** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
rng = np.random.default_rng(42)
print("Pronto. pandas", pd.__version__)

## Regressão Linear Simples · PDF p. 175

#### Exercício: Regressão simples com statsmodels

Com os dados de diabetes, ajuste `target ~ bmi` com `smf.ols`, mostre os coeficientes e o R² e desenhe a reta sobre os pontos.

In [ ]:
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame
m = smf.ols("target ~ bmi", df).fit(); print(m.params.round(3), round(m.rsquared, 3))
plt.scatter(df["bmi"], df["target"], s=8); xs = np.linspace(df["bmi"].min(), df["bmi"].max(), 2)
plt.plot(xs, m.params["Intercept"] + m.params["bmi"] * xs, "r"); plt.show()

## Regressão Linear Múltipla · PDF p. 183

#### Exercício: Regressão múltipla e RMSE

Ajuste `target ~ age + sex + bmi + bp + s1 + s5`, mostre o `summary()` e calcule o RMSE dos resíduos.

In [ ]:
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame
m = smf.ols("target ~ age + sex + bmi + bp + s1 + s5", df).fit()
print(m.summary().tables[1]); print("RMSE:", round(np.sqrt(np.mean(m.resid ** 2)), 2))

## Validação Cruzada · PDF p. 187

#### Exercício: Validação cruzada com scikit-learn

Compare, com validação cruzada de 5 lotes (RMSE), a regressão só com `bmi` e a regressão com todas as 10 variáveis.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
X, y = load_diabetes(return_X_y=True, as_frame=True)
for cols in (["bmi"], list(X.columns)):
    s = cross_val_score(LinearRegression(), X[cols], y, cv=5, scoring="neg_root_mean_squared_error")
    print(len(cols), "variáveis -> RMSE", round(-s.mean(), 2))

## Seleção de Modelo e Regressão Passo a Passo · PDF p. 188

#### Exercício: Seleção passo a passo por AIC

Implemente uma seleção *forward*: comece com `target ~ 1` e, a cada passo, acrescente a variável que mais reduz o AIC; pare quando nenhuma reduzir.

In [ ]:
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame
restantes, escolhidas = [c for c in df.columns if c != "target"], []
aic = smf.ols("target ~ 1", df).fit().aic
while restantes:
    cand = {v: smf.ols("target ~ " + " + ".join(escolhidas + [v]), df).fit().aic for v in restantes}
    v, novo = min(cand.items(), key=lambda kv: kv[1])
    if novo >= aic: break
    escolhidas.append(v); restantes.remove(v); aic = novo
print(escolhidas, round(aic, 1))

## Intervalos de Confiança e Previsão · PDF p. 193

#### Exercício: Intervalo de confiança × de previsão

No modelo `target ~ bmi`, calcule para bmi = 30 o intervalo de confiança da média e o intervalo de previsão de um paciente novo (`get_prediction(...).summary_frame(alpha=0.05)`). Por que o segundo é mais largo?

In [ ]:
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame
m = smf.ols("target ~ bmi", df).fit()
print(m.get_prediction(pd.DataFrame({"bmi": [30]})).summary_frame(alpha=0.05).round(1))

## Representação de Variáveis Fictícias · PDF p. 196

#### Exercício: Variáveis dummy

Crie um fator `faixa_idade` (jovem < 40, meia-idade < 60, idoso) e ajuste `target ~ bmi + C(faixa_idade)`. Mostre as colunas dummy com `pd.get_dummies(..., drop_first=True)` e interprete um coeficiente.

In [ ]:
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame
df["faixa_idade"] = pd.cut(df["age"], [0, 40, 60, 120], labels=["jovem", "meia_idade", "idoso"])
print(pd.get_dummies(df["faixa_idade"], drop_first=True).head())
print(smf.ols("target ~ bmi + C(faixa_idade)", df).fit().params.round(2))

## Multicolinearidade · PDF p. 203

#### Exercício: VIF

Calcule o fator de inflação da variância (VIF) das variáveis `s1` a `s6` dos dados de diabetes (`variance_inflation_factor`). Quais são muito colineares?

In [ ]:
from sklearn.datasets import load_diabetes
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = sm.add_constant(load_diabetes(as_frame=True, scaled=False).frame[["s1", "s2", "s3", "s4", "s5", "s6"]])
print({c: round(variance_inflation_factor(X.values, i), 1) for i, c in enumerate(X.columns) if c != "const"})

## Interações e Efeitos Principais · PDF p. 205

#### Exercício: Interação entre variáveis

Simule vendas = 10 + 2·anúncio + 3·promoção + 4·anúncio·promoção + ruído (anúncio contínuo, promoção 0/1). Ajuste com e sem o termo de interação e compare o R².

In [ ]:
n = 300
d = pd.DataFrame({"anuncio": rng.uniform(0, 10, n), "promocao": rng.integers(0, 2, n)})
d["vendas"] = 10 + 2 * d.anuncio + 3 * d.promocao + 4 * d.anuncio * d.promocao + rng.normal(0, 3, n)
print(round(smf.ols("vendas ~ anuncio + promocao", d).fit().rsquared, 3), round(smf.ols("vendas ~ anuncio * promocao", d).fit().rsquared, 3))

## Valores Influentes · PDF p. 210

#### Exercício: Outliers e influência

Gere uma regressão com 50 pontos e acrescente um ponto muito afastado. Calcule a distância de Cook (`get_influence().cooks_distance`) e mostre qual ponto tem a maior.

In [ ]:
x = rng.uniform(0, 10, 50); y = 2 * x + rng.normal(0, 1, 50)
x = np.append(x, 25); y = np.append(y, 5)
m = sm.OLS(y, sm.add_constant(x)).fit()
cook = m.get_influence().cooks_distance[0]; print(int(np.argmax(cook)), round(cook.max(), 2))

## Splines · PDF p. 220

#### Exercício: Polinômio × spline

Para y = sin(x) + ruído, com x de 0 a 10, ajuste um polinômio de grau 3 e uma spline cúbica (`bs(x, df=6)` na fórmula do statsmodels) e desenhe as duas curvas.

In [ ]:
d = pd.DataFrame({"x": np.sort(rng.uniform(0, 10, 200))}); d["y"] = np.sin(d.x) + rng.normal(0, 0.3, 200)
poli = smf.ols("y ~ x + I(x**2) + I(x**3)", d).fit(); spl = smf.ols("y ~ bs(x, df=6)", d).fit()
plt.scatter(d.x, d.y, s=6, c="grey"); plt.plot(d.x, poli.fittedvalues, label="polinômio 3"); plt.plot(d.x, spl.fittedvalues, label="spline"); plt.legend(); plt.show()